# Rider Counting — NEW pipeline (方法2: 逐帧检测 + 近邻帧去重)

新方法运行入口（notebook 版）。逻辑与 `scripts/run_rider_count_new.py` 完全同源（直接 import，不是复制）。

**用法**：改 Cell 2 的配置 → 依次运行。单地点跑 Cell 3–4；全量批跑用 Cell 5。

In [ ]:
# Cell 1: Bootstrap — 定位仓库根目录并加载新管线模块 (健壮版)
import sys
import importlib.util
from pathlib import Path

p = Path.cwd().resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent
assert (p / "src").exists(), "找不到包含 src/ 的仓库根目录 — 请把本 notebook 放在仓库的 notebooks/ 文件夹里"
REPO_ROOT = p
sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)

script = REPO_ROOT / "scripts" / "run_rider_count_new.py"
if not script.exists():
    hits = [h for h in REPO_ROOT.rglob("run_rider_count_new.py")]
    print("!! scripts/run_rider_count_new.py 不在预期位置")
    print("   搜索到:", hits if hits else "(整个仓库里都没有)")
    assert hits, "请把 zip 里的 scripts/ 文件夹解压到仓库根目录 (和 src/ 同级)"
    script = hits[0]
    print("   使用:", script)

spec = importlib.util.spec_from_file_location("run_rider_count_new", script)
rrc = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rrc)
print("模块加载成功:", script.name)

In [ ]:
# Cell 2: 配置 (只改这里)
from types import SimpleNamespace

DATA_ROOT = Path(r"D:\0_MAIN_BIKE_DATASETS_clean")

LOC_ID  = "loc_15"                              # 要跑的地点
IMG_DIR = DATA_ROOT / "Loc_15" / "Bicyclist"    # 该地点原始图片目录

ROI_JSON = REPO_ROOT / "configs" / "locations_new" / f"{LOC_ID}.json"
OUTDIR   = REPO_ROOT / "outputs_new" / LOC_ID

args = SimpleNamespace(
    model      = "yolov8n.pt",   # 召回排查可换 "yolov8s.pt"/"yolov8m.pt"
    imgsz      = 640,            # 召回排查可提到 1280 (小目标/夜间)
    conf       = 0.25,          # 显式传给 YOLO (旧版0.10未生效, 实际就是0.25)
    classes    = {1},           # COCO bicycle
    nms_iou    = 0.70,
    assoc_gap  = 3,             # 编号差<=3 的近邻帧做同人关联; 更远=不同人
    min_move_px= 8.0,
    cos_gate   = 0.5,           # |cos|低于此值=横穿, 不判WW
    save_crops = True,          # 导出每个rider裁剪图, 供朝向标注
    max_images = None,          # 调试时可设 200
)

print("LOC_ID:", LOC_ID)
print("IMG_DIR:", IMG_DIR, "| exists:", IMG_DIR.exists())
print("ROI_JSON:", ROI_JSON, "| exists:", ROI_JSON.exists())

In [ ]:
# Cell 3: 跑当前地点
summary = rrc.run_location(LOC_ID, IMG_DIR, ROI_JSON, OUTDIR, args)
summary

In [ ]:
# Cell 4: 查看结果
import pandas as pd

riders = pd.read_csv(OUTDIR / "riders.csv")
print(f"riders: {len(riders)} 人 | 多帧关联: {(riders.n_obs>=2).sum()} | 单帧: {(riders.n_obs==1).sum()}")
print("\n设施分布 (any-involvement):")
print("  sidewalk :", int(riders.in_sidewalk_any.sum()))
print("  bike_lane:", int(riders.in_bike_lane_any.sum()))
print("  roadway  :", int(riders.in_roadway_any.sum()))
dk = riders[riders.direction_displacement.isin(["along_flow","against_flow"])]
n_cross = int((riders.direction_displacement == "cross_flow").sum())
if len(dk):
    ww = int((dk.wrong_way_displacement == True).sum())
    print(f"\n沿街方向已知: {len(dk)} | wrong-way: {ww} ({ww/len(dk)*100:.0f}%) | 横穿(不计WW): {n_cross}")
riders.head(10)

In [ ]:
# Cell 5: (可选) 全量批跑 — 遍历 configs/locations_new/ 下所有地点
RUN_BATCH = False   # 确认单地点没问题后改成 True 再运行本cell

if RUN_BATCH:
    import pandas as pd
    summaries = []
    cfg_dir = REPO_ROOT / "configs" / "locations_new"
    out_root = REPO_ROOT / "outputs_new"
    for cfg in sorted(cfg_dir.glob("loc_*.json")):
        if "old" in cfg.stem.lower():
            continue
        loc = cfg.stem
        img_dir = rrc.find_img_dir(DATA_ROOT, loc)
        if img_dir is None:
            print(f"[{loc}] 找不到图片文件夹, 跳过")
            continue
        s = rrc.run_location(loc, img_dir, cfg, out_root / loc, args)
        if s:
            summaries.append(s)
    df = pd.DataFrame(summaries)
    df.to_csv(out_root / "all_locations_summary.csv", index=False)
    print(f"\n批跑完成: {len(summaries)} 个地点")
    display(df)

In [ ]:
# Cell 6: (可选) 批跑结果一览图
import pandas as pd
f = REPO_ROOT / "outputs_new" / "all_locations_summary.csv"
if f.exists():
    import matplotlib.pyplot as plt
    df = pd.read_csv(f).sort_values("n_riders", ascending=False)
    ax = df.set_index("location_id")[["riders_in_sidewalk","riders_in_bike_lane","riders_in_roadway"]].plot(
        kind="bar", figsize=(14,4), color=["#00bcd4","#2e9e46","#d62728"])
    ax.set_ylabel("riders"); ax.set_title("Riders per facility (NEW pipeline)")
    plt.tight_layout(); plt.show()
else:
    print("还没有批跑结果")